# ch10 分组聚合：`agg` 的四种写法和三个默认值陷阱

> 方向：数据预处理 ｜ 竞赛对应：**数据挖掘（分组统计）+ 数据探索（分布概览）**
> —— 分组聚合是"从行级数据里抽特征"的标准手段，
> 也是把「明细表」变成「一张能看的汇总表」的唯一办法。

本章的核心不是"会用 `groupby`"，而是三件更硬的事：

1. `agg` 有**四种写法**，输出的列结构各不相同——选错就得返工重写
2. `size()` 和 `count()` **差 21 行**，这两个最容易被当成同一个东西
3. `groupby` 有**三个默认值**会在你不知情时改变结果：`dropna` / `observed` / `as_index`

## 一、本节考点

| 竞赛评分点 | 分值含义 | 本节覆盖的操作 |
|---|---|---|
| 数据挖掘 · 分组统计 | 20 分模块内的核心动作 | 按维度聚合指标、交叉汇总 |
| 数据探索 · 分布概览 | 每项 2 分 | 分组后的均值/中位数/最大最小 |
| 特征工程 | 影响建模分 | 组内均值/标准差/排名广播回每一行 |
| 结果交付 | 提交格式分 | `as_index=False` 拿到可直接写出的表 |

**为什么单独成章**：竞赛里的"数据挖掘"环节几乎都是从
「按台区/按设备类型/按时间维度汇总」开始的。
而 `groupby` 是 pandas 里**参数默认值最多、且默认值会改结果**的 API——
本章会把三个默认值逐一实测出来。

## 二、API 速查表

| API | 返回 | 一句话说明 |
|---|---|---|
| `df.groupby(键)` | `DataFrameGroupBy` | 只分组，不算（惰性对象） |
| `gb.size()` | `Series` | 每组**行数**（含缺失，不排除） |
| `gb[列].count()` | `Series` | 每组该列的**非缺失**个数 |
| `gb[列].mean()/sum()/median()/std()` | `Series` | 单函数快捷聚合 |
| `gb[列].agg(["mean","max"])` | `DataFrame` | 多函数，列名 = 函数名 |
| `gb.agg({"列":["mean"],"列2":"sum"})` | `DataFrame` | 多列多函数，**列名变 MultiIndex** |
| `gb.agg(别名=("列","函数"))` | `DataFrame` | **命名聚合**（推荐，列名直接可用） |
| `gb.transform("mean")` | `Series` | 组统计量**广播回原长度**（见 ch11） |
| `gb.rank(pct=True)` | `Series` | 组内排名（0~1） |
| `gb.head(n)` | `DataFrame` | 每组前 n 行（配合 `sort_values` 做组内 top N） |
| `gb.ngroups` | `int` | 组数 |
| `df.pivot_table(index=,columns=,values=,aggfunc=)` | `DataFrame` | 交叉汇总（等价于"多维 groupby + unstack"） |
| `pd.crosstab(行,列)` | `DataFrame` | 交叉**计数**（只数条数） |

### `agg` 四种写法对照（最重要的表）

| 写法 | 代码 | 输出列名 | 什么时候用 |
|---|---|---|---|
| ① 单函数 | `gb["负荷值"].mean()` | `Series`，index=组 | 只要一个指标 |
| ② 函数列表 | `gb["负荷值"].agg(["mean","max"])` | 函数名（单层） | 同一列多个指标 |
| ③ 字典 | `gb.agg({"负荷值":["mean"],"湿度":"mean"})` | **MultiIndex 元组** | 多列多指标（竞赛少用） |
| ④ 命名聚合 | `gb.agg(均值=("负荷值","mean"))` | **自定义名（单层）** | ⭐ **默认选它** |

> ④ 为什么最好：`as_index=False` + 命名聚合能直接得到一张
> **列名正常、可以 `to_csv` 直接交卷**的表。② ③ 的列名要么是函数名，
> 要么是元组，后续还得改列名——多一步就多一个出错点。

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "STHeiti"]
plt.rcParams["axes.unicode_minus"] = False

DATA = Path("data")
dev = pd.read_csv(DATA / "device_defects.csv")

GROUP_KEY = "设备类型"
VALUE_COL = "负荷值"

print("pandas", pd.__version__)
print("dev", dev.shape)
print("分组基准：", dev.groupby(GROUP_KEY).size().to_dict())

---

## 三、逐节讲解

### 3.1 心智模型：split → apply → combine

`groupby` 干三件事，记住这三步，所有参数都能对上号：

| 阶段 | 做什么 | 对应参数 |
|---|---|---|
| **split** 分组 | 按键把行切成若干块 | `dropna`（缺失键算不算一组） / `observed`（空类别算不算一组） |
| **apply** 计算 | 在每块上跑聚合函数 | `agg` / `transform` / `apply` |
| **combine** 合并 | 把结果拼回一张表 | `as_index`（组键变索引还是变列） / `sort` |

**"缺一块行不行"是判断参数归属的最好办法**：凡是影响"行被分到哪块"的，都是 split 阶段；
凡是影响"结果表长什么样"的，都是 combine 阶段。

In [ ]:
gb = dev.groupby(GROUP_KEY)
print("① groupby 是惰性的——只分组，不算：")
print("   对象类型:", type(gb).__name__)
print("   组数 ngroups:", gb.ngroups)
print("   组键:", list(gb.groups.keys()))

print("\n② 各组行数 size()：")
size_g = gb.size()
print(size_g.to_string())
print("   合计:", int(size_g.sum()), "= 行数", len(dev))

print("\n③ count() 与 size() 的差别（这里就出问题了）：")
count_g = gb[VALUE_COL].count()
cmp_df = pd.DataFrame({"size": size_g, "count": count_g})
cmp_df["差（=缺失数）"] = cmp_df["size"] - cmp_df["count"]
print(cmp_df.to_string())
print("\n   size 合计:", int(size_g.sum()))
print("   count 合计:", int(count_g.sum()), " ← 少", int(size_g.sum() - count_g.sum()), "个，正是负荷值的缺失数")
print("   -> size() 数的是行，count() 数的是值。'多少台设备'用 size，'多少条有效负荷记录'用 count")

> **判定规则**：
> - 问"**有多少条记录**" → `size()`
> - 问"**这个指标有多少个有效观测**" → `count()`
>
> 这两个之所以危险，是因为**数据没有缺失时它们完全相等**，
> 于是你会在小样本上验证通过、在正式数据上差 21 行。
> 写 `agg` 时同理：`("负荷值", "size")` 和 `("负荷值", "count")` 是两个不同的指标。

---

### 3.2 单函数聚合：五个最常用的

`mean` / `sum` / `median` / `std` / `max` 都可以直接挂。但要记住一件事：
**`mean` 会被异常值污染**——本章数据里负荷值含 7 个 `>= 1000` 的哨兵码，
所以分组均值会明显偏高。所以下面每一组都同时输出 `mean` 和 `median`。

In [ ]:
print("① 分组均值（先看这个，但别全信）：")
mean_g = dev.groupby(GROUP_KEY)[VALUE_COL].mean()
print(mean_g.round(4).to_string())

print("\n② 均值 vs 中位数：哨兵值把均值拉高了")
med_g = dev.groupby(GROUP_KEY)[VALUE_COL].median()
cmp2 = pd.DataFrame({"mean": mean_g, "median": med_g})
cmp2["mean/median"] = (cmp2["mean"] / cmp2["median"]).round(2)
print(cmp2.round(2).to_string())
print("\n   变压器组：mean = %.2f，median = %.2f，相差 %.1f 倍"
      % (mean_g["变压器"], med_g["变压器"], mean_g["变压器"] / med_g["变压器"]))
print("   原因：该组里有 9999 这种哨兵码，把均值拽上去了")

print("\n③ 其他常用聚合：")
print("   合计:", dev.groupby(GROUP_KEY)[VALUE_COL].sum().round(2).to_dict())
print("   标准差:", dev.groupby(GROUP_KEY)[VALUE_COL].std().round(2).to_dict())
print("   最小值:", dev.groupby(GROUP_KEY)[VALUE_COL].min().to_dict(), " ← 3 个组的最小值是 -1 哨兵")
print("   湿度均值:", dev.groupby(GROUP_KEY)["湿度"].mean().round(2).to_dict())

> **一个能立刻提升分析质量的习惯**：任何"分组求均值"的地方，
> **旁边都放一个中位数**。两者差距大 → 组内有异常值 → 先回去清洗。
> 本章数据里变压器组 `mean/median = 1.89`，这就是明确的清洗信号。

---

### 3.3 `agg` 的四种写法与它们的输出结构

这一节是本章的核心。四种写法的**输出结构完全不同**，选错不会报错，
但你后面 `to_csv` 或 `merge` 的时候才发现列名是一堆元组。

In [ ]:
gv = dev.groupby(GROUP_KEY)[VALUE_COL]

print("① 单函数 -> Series（index 是组键）")
s1 = gv.mean()
print("   type:", type(s1).__name__, "| index.name:", s1.index.name)

print("\n② 函数列表 -> DataFrame，列名是函数名（单层）")
s2 = gv.agg(["count", "mean", "median", "std", "max"])
print("   shape:", s2.shape, "| columns:", list(s2.columns))
print(s2.round(2).to_string())

print("\n③ 字典 -> DataFrame，但列名是 MultiIndex 元组（最容易踩的坑）")
s3 = dev.groupby(GROUP_KEY).agg({"负荷值": ["mean", "max"], "湿度": "mean"})
print("   shape:", s3.shape)
print("   columns 类型:", type(s3.columns).__name__)
print("   columns 内容:", list(s3.columns))
print("   -> 想把它压成单层列名，要么 s3.columns = [...] 手工改，要么直接用命名聚合")

print("\n④ 命名聚合 -> 列名完全由你定（推荐）")
s4 = dev.groupby(GROUP_KEY).agg(
    样本数=("负荷值", "size"),
    有效负荷数=("负荷值", "count"),
    平均负荷=("负荷值", "mean"),
    中位负荷=("负荷值", "median"),
    最大负荷=("负荷值", "max"),
    危急数=("缺陷等级", lambda s: int((s == "危急").sum())),
)
print("   shape:", s4.shape, "| columns:", list(s4.columns))
print(s4.round(2).to_string())

> **命名聚合里可以用 lambda**，这让它比字典写法强得多：任何"自定义指标"
> （危急率、超标条数、去重计数……）都能塞进同一张表。
>
> **注意**：`agg` 里的 lambda 收到的是**每个组的子 Series**，
> 所以要写 `lambda s: int((s == "危急").sum())`——拿到的是值，不是行。

---

### 3.4 难点深挖：`as_index` —— 结果表能不能直接交卷

**为什么难**：`groupby` 默认把组键变成**索引**（`as_index=True`）。
这让 `df["设备类型"]` 这种写法失效（因为它是 index 不是列），
于是你要么 `reset_index()`，要么一开始就设 `as_index=False`。
**两个选择都没错，但混用会让代码里到处是 `reset_index`。**

In [ ]:
print("① 默认 as_index=True：组键是索引，不是列")
d1 = dev.groupby(GROUP_KEY).agg(平均负荷=("负荷值", "mean"))
print("   columns:", list(d1.columns))
print("   index.name:", d1.index.name)
print("   能写 d1['设备类型'] 吗:", "设备类型" in d1.columns, " ← 不在列里")

print("\n② as_index=False：组键变回普通列")
d2 = dev.groupby(GROUP_KEY, as_index=False).agg(
    样本数=("负荷值", "size"),
    平均负荷=("负荷值", "mean"),
)
print("   columns:", list(d2.columns))
print(d2.round(2).to_string(index=False))

print("\n③ 两种写法数值上完全一致（只是组键位置不同）：")
d3 = d1.reset_index()
print("   形状:", d3.shape, "vs", d2.shape)
print("   平均负荷数值一致:", bool(np.allclose(d3["平均负荷"].to_numpy(), d2["平均负荷"].to_numpy())))
print("   组键顺序一致:", d3[GROUP_KEY].tolist() == d2[GROUP_KEY].tolist())

print("\n④ 建议：")
print("   要交卷 / 要 to_csv / 要 merge -> 一律 as_index=False + 命名聚合")
print("   只做中间计算、还要 .loc[组名] 取数 -> 用默认的 as_index=True 更方便")

> **判定规则**：
> - 结果是**最终产出**（要写文件、要 join、要画图）→ `as_index=False`
> - 结果是**中间步骤**（还要 `.loc["变压器"]` 取某个组）→ 保留索引更方便
>
> 一条纪律：**一个 notebook 里选定一种风格就别改**。混用会出现
> "有时候 `df['设备类型']` 能用、有时候不能用"的困惑，而这类困惑
> 在限时竞赛里是最浪费时间的。

---

### 3.5 难点深挖：`groupby` 的三个默认值（都会改结果）

`groupby` 最反直觉的地方在于：**有三个默认值会静默改变你的结果**，
而且它们的默认值在 pandas 版本间还变过。

| 参数 | 本机 3.0 默认 | 作用 | 踩坑表现 |
|---|---|---|---|
| `dropna` | `True` | 键是缺失的行**整组丢掉** | 组数莫名少一组，且**不报警** |
| `observed` | **`True`**（2.x 是 `False`） | 类别键只保留"实际出现过的"类别 | 老教程教你设 `True`，3.0 已默认 `True` |
| `as_index` | `True` | 组键变索引 | `df["组键"]` 报 KeyError |

In [ ]:
print("① dropna 默认 True：键缺失的整组被丢掉")
d = dev.copy()
d.loc[0:4, "缺陷等级"] = np.nan          # 人为制造 5 行缺失键
print("   默认 dropna=True :", d.groupby("缺陷等级").size().to_dict())
print("   dropna=False     :", d.groupby("缺陷等级", dropna=False).size().to_dict())
print("   -> 少了 nan 那一组，而且不报错。'各组加起来等于总行数'这个自检能抓住它")
print("   自检: 默认分组求和 =", int(d.groupby("缺陷等级").size().sum()),
      " vs 总行数 =", len(d))

print("\n② observed 的默认值在 3.0 变了（这条会让老教程把你带偏）")
full_dtype = pd.CategoricalDtype(categories=dev["设备类型"].astype("category").cat.categories)
sub = dev[dev["设备类型"].isin(["变压器", "绝缘子"])]
key = sub["设备类型"].astype(full_dtype)
print("   子集里只有 2 类，但 key 的 categories 有", len(full_dtype.categories), "类")
print("   不传 observed :", sub.groupby(key).size().to_dict(), " ← 2 组")
print("   observed=True :", sub.groupby(key, observed=True).size().to_dict(), " ← 与上面一致")
obs_false = sub.groupby(key, observed=False).size()
print("   observed=False:", obs_false.to_dict())
print("   -> 组数变成", len(obs_false), "，多出 3 个计数为 0 的空组（笛卡尔积）")
print("   实测默认值：", "True（3.0 起）" if sub.groupby(key).size().shape[0] == 2 else "False")

print("\n③ as_index 影响的是形状，不是数值")
print("   见上一节，这里只提醒：它不会改数值，只会改组键的位置")

print("\n④ 三条自检（写进每个分组任务里）")
print("   1) gb.size().sum() == len(df)                 -> 抓 dropna")
print("   2) len(gb.size()) == df[键].nunique()          -> 抓 observed 空组")
print("   3) '组键' in result.columns                    -> 抓 as_index")

> **判定规则**（背下来）：
>
> | 场景 | 怎么写 |
> |---|---|
> | 键**确定没有缺失** | 用默认，不用管 |
> | 键**可能有缺失**且缺失本身有含义（"未评级"） | `dropna=False` |
> | 键是 `category` 且只统计实际出现的类别 | 用默认（3.0 已是 `observed=True`） |
> | 要把**所有类别**（含未出现的）都列出来 | `observed=False`（做完整报表时才需要） |
> | 结果要交卷 | `as_index=False` |
>
> **每次分组后立刻跑那三条自检**，比事后 debug 便宜一百倍。

---

### 3.6 多键分组与 `unstack`

按两个维度分组会得到 MultiIndex；`unstack` 把最内层搬到列上，
就变成一张**交叉表**——这是竞赛里"一次性看清楚"的标准图形。

In [ ]:
print("① 两个键分组 -> MultiIndex")
gm = dev.groupby(["台区编号", "设备类型"])[VALUE_COL].mean()
print("   组数:", len(gm), "| index.names:", gm.index.names)
print("   前 4 行:")
print(gm.head(4).round(2).to_string())

print("\n② unstack 把最内层键搬到列上 -> 交叉表")
cross = gm.unstack()
print("   形状:", cross.shape, "（18 台区 x 5 类型）")
print(cross.round(1).iloc[:4, :].to_string())
print("\n   空单元格（NaN）的含义：那个台区没有那种设备 -> 这是真·缺失，不是数据错误")

print("\n③ 台区填满类型：")
print("   每个台区有几种设备:", cross.notna().sum(axis=1).value_counts().sort_index().to_dict())

print("\n④ as_index=False + 多键，直接得到长表")
gl = dev.groupby(["台区编号", "设备类型"], as_index=False)[VALUE_COL].mean()
print("   columns:", list(gl.columns), "| shape:", gl.shape)

print("\n⑤ 交叉计数用 crosstab 更直接")
ct = pd.crosstab(dev["设备类型"], dev["缺陷等级"])
print("   shape:", ct.shape)
print(ct.to_string())
print("\n   行合计:", ct.sum(axis=1).to_dict())

> **`unstack()` 之后的 `NaN` 是什么**：是"这个组合不存在"，不是"数据缺失"。
> 两者必须分开对待——**填 0 还是留 `NaN` 取决于业务**：
> - "该台区没有这种设备" → 填 0 会误导（0 台和没有是两回事），留 `NaN` 更好
> - "该台区这种设备的条数" → 填 0 才对（真的 0 条）

---

### 3.7 组内选取：`sort_values` + `groupby().head(n)`

"每个组取最大的 N 条"是竞赛高频题。标准套路是**先全局排序，再分组取头**。

In [ ]:
print("① 每组取负荷值最高的 1 条：")
top1 = dev.sort_values(VALUE_COL, ascending=False).groupby(GROUP_KEY).head(1)
print(top1[["记录ID", GROUP_KEY, VALUE_COL, "缺陷等级"]].to_string(index=False))

print("\n② 注意：三个组的 top1 都是 9999 —— 哨兵码污染了极值选取")
print("   -> 取 top N 之前必须先把哨兵处理掉（回看 ch05）")

print("\n③ 清洗后再取：")
dev_clean = dev[dev[VALUE_COL].between(0, 1000)]
top1_clean = dev_clean.sort_values(VALUE_COL, ascending=False).groupby(GROUP_KEY).head(1)
print(top1_clean[["记录ID", GROUP_KEY, VALUE_COL]].to_string(index=False))

print("\n④ 每组取前 2 条（按负荷值）：")
top2 = dev_clean.sort_values(VALUE_COL, ascending=False).groupby(GROUP_KEY).head(2)
print("   形状:", top2.shape, "= 5 组 x 2 条")
print("   组内排名 rank(pct=True) 前 5 行:",
      dev.groupby(GROUP_KEY)[VALUE_COL].rank(pct=True).head(5).round(3).tolist())

print("\n⑤ 每组最小 1 条（升序排序后取 head）：")
bot1 = dev_clean.sort_values(VALUE_COL).groupby(GROUP_KEY).head(1)
print(bot1[["记录ID", GROUP_KEY, VALUE_COL]].to_string(index=False))

> **顺序不能反**：`groupby().head(n)` 是"取每组**原始顺序**里的前 n 行"，
> **它不会自动排序**。所以一定是 `sort_values(...)` 在前。
> 反过来的写法（`groupby().apply(lambda d: d.nlargest(n, col))`）结果对但慢很多，
> 且返回结构会变（见 ch11）。

---

### 3.8 `pivot_table` 与 `groupby` 的等价关系

`pivot_table` 不是新东西，它就是"多维 `groupby` + `unstack`"的语法糖。
用数据验证这个等价关系，你以后就知道什么时候用哪个。

In [ ]:
print("① pivot_table 写法：")
pt = dev.pivot_table(index=GROUP_KEY, columns="缺陷等级", values=VALUE_COL, aggfunc="mean")
print("   shape:", pt.shape)
print(pt.round(1).to_string())

print("\n② 等价的 groupby 写法：")
gt = dev.groupby([GROUP_KEY, "缺陷等级"])[VALUE_COL].mean().unstack()
print("   两者形状相同:", pt.shape == gt.shape)
print("   逐格数值一致:", bool(np.allclose(pt.to_numpy(), gt.to_numpy(), equal_nan=True)))

print("\n③ 抽一格手工验证：变压器 x 危急")
manual = dev[(dev[GROUP_KEY] == "变压器") & (dev["缺陷等级"] == "危急")][VALUE_COL].mean()
print("   pivot_table:", round(pt.loc["变压器", "危急"], 4))
print("   手写筛选  :", round(manual, 4))
print("   一致:", abs(pt.loc["变压器", "危急"] - manual) < 1e-9)

print("\n④ aggfunc='count' 时是计数表：")
print(dev.pivot_table(index=GROUP_KEY, columns="缺陷等级", values="记录ID", aggfunc="count").to_string())
print("   与 crosstab 一致?",
      dev.pivot_table(index=GROUP_KEY, columns="缺陷等级", values="记录ID", aggfunc="count").equals(
          pd.crosstab(dev[GROUP_KEY], dev["缺陷等级"])))

> **选型规则**：
> - 结果要**继续参与计算**（还要 `merge`、还要加减） → 用 `groupby`（能拿到标准列名）
> - 结果是**给人看的报表**，两个维度交叉 → 用 `pivot_table`（可读性更好）
> - 只数**条数** → 用 `pd.crosstab`（最简洁）
>
> `pivot_table` 的默认 `aggfunc` 是 `"mean"`——**这是它最大的暗坑**：
> 忘记写 `aggfunc` 时它会静默求均值，而你以为是计数。

---

### 3.9 `transform`：把组统计量广播回每一行（ch11 预告）

`agg` 把 270 行压成 5 行；`transform` **保持 270 行**，
把每组的统计量贴回该组每一行。这是特征工程里最常用的一招：
"这条记录的负荷值，相对于它所在设备类型是高还是低？"

In [ ]:
print("① agg 会压缩行数，transform 不会：")
print("   agg 长度      :", len(dev.groupby(GROUP_KEY)[VALUE_COL].mean()))
print("   transform 长度:", len(dev.groupby(GROUP_KEY)[VALUE_COL].transform("mean")))

print("\n② transform 的值就是该行所属组的统计量：")
dev_feat = dev.copy()
dev_feat["组均值"] = dev_feat.groupby(GROUP_KEY)[VALUE_COL].transform("mean")
dev_feat["组标准差"] = dev_feat.groupby(GROUP_KEY)[VALUE_COL].transform("std")
print(dev_feat[["记录ID", GROUP_KEY, VALUE_COL, "组均值"]].head(5).round(2).to_string(index=False))
print("\n   第 0 行属于", dev_feat.loc[0, GROUP_KEY], "，组均值 =", round(dev_feat.loc[0, "组均值"], 4))

print("\n③ 组内 z-score：这条记录相对同类型设备偏了多少个标准差")
dev_feat["组内z"] = (dev_feat[VALUE_COL] - dev_feat["组均值"]) / dev_feat["组标准差"]
print("   前 3 行 z:", dev_feat["组内z"].head(3).round(3).tolist())

print("\n④ 组内百分位排名：")
dev_feat["组内排名"] = dev_feat.groupby(GROUP_KEY)[VALUE_COL].rank(pct=True)
print("   前 5 行:", dev_feat["组内排名"].head(5).round(3).tolist())

print("\n⑤ 组内占比：这条记录的负荷占本组总负荷的比例")
dev_feat["组内占比"] = dev_feat[VALUE_COL] / dev_feat.groupby(GROUP_KEY)[VALUE_COL].transform("sum")
print("   各组组内占比之和:", dev_feat.groupby(GROUP_KEY)["组内占比"].sum().round(6).to_dict())
print("   -> 每组都加起来等于 1.0，这就是 transform 广播正确的证明")

---

## 四、易错点清单

1. **`size()` 当成 `count()`** → 有缺失时差 21 行，且无缺失时完全相等（发现不了）
2. `agg` 用字典写法 → 列名变成 MultiIndex 元组，`to_csv` 出来是一堆括号
3. `agg` 后想用 `df["组键"]` → 报 `KeyError`（默认 `as_index=True`，组键在索引上）
4. 分组后**不做"各组之和 == 总行数"自检** → 漏掉 `dropna` 丢的那一组
5. `dropna` 默认 `True` → 键缺失的行整组消失，**不报警**
6. `observed` 在 3.0 默认已是 `True` → 照抄老教程显式传 `True` 是多余（但无害）；
   真要看空类别得显式传 `False`
7. `unstack()` 后的 `NaN` 被一律填 0 → "该组合不存在"和"计数为 0"被混淆
8. **`groupby().head(n)` 不排序** → 拿到的是原始顺序前 n 行，不是最大 n 行
9. 取组内 top N 前不清哨兵 → 三个组的 top1 全是 9999（实测）
10. 只看 `mean` 不看 `median` → 异常值污染看不出来（变压器组 mean/median = 1.89）
11. `pivot_table` 忘记 `aggfunc` → 默认求**均值**，你以为在做计数
12. `transform` 与 `agg` 混用 → 一个保持长度、一个压缩长度，写错必炸

---

## 五、本章小结

| 需求 | 正确写法 |
|---|---|
| 每组有多少条记录 | `gb.size()` |
| 每组某列有多少有效值 | `gb[列].count()` |
| 一列多指标 | `gb[列].agg(["mean","max"])` |
| 多列多指标（**推荐**） | `gb.agg(均值=("负荷值","mean"), 危急数=("缺陷等级", lambda s: ...))` |
| 结果要交卷 | 加 `as_index=False`，得到正常列名的表 |
| 键可能有缺失 | `dropna=False`（否则整组消失） |
| 要看未出现的类别 | `observed=False`（3.0 默认已是 `True`） |
| 二维交叉表 | `pivot_table(index=,columns=,values=,aggfunc=)` 或 `crosstab` |
| 组内 top N | `df.sort_values(列, ascending=False).groupby(键).head(n)` |
| 组统计量广播回每行 | `gb[列].transform("mean")` |
| 三条自检 | `size().sum()==len(df)`、`组数==nunique()`、`'组键' in result.columns` |